In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.io import  wavfile
from IPython import display
import json, os

In [ ]:
import numpy as np
import matplotlib.pyplot as plt


# Compute normalized cross-correlation
def normalized_cross_correlation(x1, x2):
    n1 = len(x1)
    result = []
    for i in range(len(x2) - n1 + 1):
        segment = x2[i:i + n1]
        numerator = np.sum((x1 - np.mean(x1)) * (segment - np.mean(segment)))
        denominator = np.sqrt(np.sum((x1 - np.mean(x1))**2) * np.sum((segment - np.mean(segment))**2))
        result.append(numerator / denominator if denominator != 0 else 0)
    return np.array(result)

def find_sub_array(target, long_array):
  correlation = normalized_cross_correlation(target, long_array)
  
  # Find the starting index where target best matches long_array
  best_start_index = np.argmax(correlation)
  
  # Plot normalized cross-correlation
  # plt.figure(figsize=(12, 5))
  plt.subplot(2, 1, 1)
  plt.plot(correlation, label='Normalized Cross-correlation')
  plt.axvline(best_start_index, color='r', linestyle='--', label=f'Best match: {best_start_index}')
  plt.title('Normalized Cross-correlation between target and long_array')
  plt.legend()
  
  # Plot long_array and overlay target at the best match position
  plt.subplot(2, 1, 2)
  plt.plot(long_array, label='long_array', marker='.', color='gray', alpha=0.7)
  plt.plot(
      range(best_start_index, best_start_index + len(target)),
      target,
      label='target (best match)',
      marker='o',
      color='red'
  )
  plt.title('long_array with target overlayed at best match position')
  plt.legend()
  plt.tight_layout()
  rms_error = np.sqrt(np.mean((long_array[best_start_index:best_start_index + len(target)] - target)**2))
  print(f"RMS Error of sub-string = {rms_error}")


# Example data
plt.figure(figsize=(6,4))
x_long = np.random.rand(100)  # Longer array
x_short = x_long[62:75] # + np.random.normal(0, 0.05, len(x_short))  # Shorter array, with some noise
find_sub_array(x_short, x_long)


In [ ]:
# wavfile.write?
# wavfile.read?
fs, wav_from_file = wavfile.read("/Users/jeremy/tmp/happy_2ch.wav")
print(f"fs={fs}, shape = {wav_from_file.shape}")

In [ ]:
pwd

In [ ]:
wav_dir = os.getenv("HOME")
wav_dir = "."
with open(os.path.join(wav_dir, "wav.json"), "r") as fpi:
  recorded_wav_stereo = json.load(fpi)
  recorded_wav_stereo = np.array(recorded_wav_stereo)
  

In [ ]:
print(recorded_wav_stereo[:12])
print(wav_from_file[:12,0])


In [ ]:
print(f"Change the 4 to 2.  This is just temporary debugging!!")
num_channels = 4
num_samples = len(recorded_wav_stereo)//num_channels
recorded_wav = np.reshape(recorded_wav_stereo[0:num_channels*num_samples], (num_samples, num_channels))
print(f"recorded wav shape = {recorded_wav.shape}")

In [ ]:
num_samples = recorded_wav.shape[0]

In [ ]:

plt.subplot(3,1,1)
plt.plot(wav_from_file[:num_samples,0])
plt.subplot(3,1,2)
plt.plot(recorded_wav[:num_samples,0])
plt.subplot(3,1,3)
plt.plot(wav_from_file[:num_samples,0] - recorded_wav[:num_samples,0])

In [ ]:
find_sub_array(recorded_wav[0:128,0], wav_from_file[:,0])

In [ ]:
find_sub_array(recorded_wav[129:256,0], wav_from_file[:,0])

In [ ]:
find_sub_array(recorded_wav[384:512,0], wav_from_file[:,0])

In [ ]:
x = np.array([0x00, 0xFF, 0xAA, 0x55])

In [ ]:
plt.clf()
k=0
plt.plot(np.arange(0,17), recorded_wav[k*16:k*16+17,0], 'r-o')
# plt.plot(np.arange(0,17), wav_from_file[2*k*16:2*k*16+17,0], 'b-+')
plt.plot(np.arange(0,17), wav_from_file[k*16:k*16+17,0], 'b-+')
plt.grid(True)

In [ ]:
0xFFE8

In [ ]:
def hex_to_signed_decimal(hex_value):
    # Convert the hex string to an integer
    num = int(hex_value, 16)
    # Check if the number is negative (16-bit signed range: -32768 to 32767)
    if num >= 0x8000:  # If the most significant bit is set
        num -= 0x10000  # Convert to negative using 2's complement
    return num

In [ ]:
hex_to_signed_decimal("0x5555")

In [ ]:
f"{512:X}"

In [ ]:
wav_from_file[:10]

In [ ]:
print(f"b0={0x1910}, b1={0x1d18}, wr={0x2120}")

In [ ]:
0x1d18+1024

In [ ]:
0x1d18

In [ ]:
def generate_sine_wave(frequency=200, sample_rate=16000, sig_len=2048, amplitude=1.0):
    """
    Generate a sine wave and write to a C array.
    
    Args:
        frequency (float): Frequency of the sine wave in Hz.
        sample_rate (int): Sampling rate in samples per second (Hz).
        duration (float): Duration of the sine wave in seconds.
        amplitude (int): Maximum amplitude of the sine wave (for 16-bit signed values).

    Returns:
        None
    """
    # Generate time values
    t = np.arange(0, int(sig_len))/sample_rate
    
    # Generate sine wave
    sine_wave = (amplitude * np.sin(2 * np.pi * frequency * t)).astype(np.float32)
    # Create the C array string
    c_array_str = f"const float32_t sine_wave[{len(sine_wave)}] = {{\n"
    c_array_str += ",\n".join(
        ", ".join(f"{x:5.4}" for x in sine_wave[i:i+8]) 
        for i in range(0, len(sine_wave), 8)
    )
    c_array_str += "\n};\n"
    
    # Write the C array to a file
    # with open("sine_wave_array.h", "w") as f:
    print(c_array_str)

    # print("C array written to sine_wave_array.h")

# Generate and save the sine wave
generate_sine_wave(frequency=7992)

In [ ]:
def generate_cmplx_sine_wave(frequency=200, sample_rate=16000, sig_len=1024, amplitude=1.0):
    """
    Generate a sine wave and write to a C array.  Array will be complex but with all imaginary values
    set to 0.  Structured as [x_real_0, x_imag_0, x_real_1, x_imag_1, ....]
    
    Args:
        frequency (float): Frequency of the sine wave in Hz.
        sample_rate (int): Sampling rate in samples per second (Hz).
        sig_len (int): number of time samples.  Output array will be 2x this length
        amplitude (int): Maximum amplitude of the sine wave (for 16-bit signed values).

    Returns:
        None
    """
    # Generate time values
    t = np.arange(0, int(sig_len))/sample_rate
    
    # Generate sine wave
    sine_wave = (amplitude * np.sin(2 * np.pi * frequency * t)).astype(np.float32)
    sine_wav_cplx = np.zeros(2*sig_len)
    sine_wav_cplx[0:-1:2] = sine_wave
    # Create the C array string
    c_array_str = f"const float32_t sine_wave[{len(sine_wav_cplx)}] = {{\n"
    c_array_str += ",\n".join(
        ", ".join(f"{x:5.4}" for x in sine_wav_cplx[i:i+8]) 
        for i in range(0, len(sine_wav_cplx), 8)
    )
    c_array_str += "\n};\n"
    
    # Write the C array to a file
    # with open("sine_wave_array.h", "w") as f:
    print(c_array_str)

    # print("C array written to sine_wave_array.h")

# Generate and save the sine wave
generate_cmplx_sine_wave(frequency=7992)

In [ ]:
fft_mag = [9.2688, 9.3713, 9.6853, 10.2312, 11.0485, 12.2054, 13.8180, 16.0908, 
19.4064, 24.5580, 33.4815, 52.4252, 118.8994, 479.8513, 80.7389, 44.4752, 
30.8839, 23.7673, 19.3879, 16.4202, 14.2751, 12.6512, 11.3782, 10.3527, 
9.5083, 8.8004, 8.1980, 7.6787, 7.2262, 6.8281, 6.4749, 6.1594, 
5.8755, 5.6188, 5.3852, 5.1718, 4.9760, 4.7955, 4.6282, 4.4749, 
4.3306, 4.1963, 4.0707, 3.9529, 3.8423, 3.7381, 3.6398, 3.5469, 
3.4589, 3.3755, 3.2963, 3.2209, 3.1492, 3.0807, 3.0154, 2.9529, 
2.8932, 2.8359, 2.7810, 2.7283, 2.6777, 2.6291, 2.5823, 2.5372, 
2.4949, 2.4518, 2.4114, 2.3723, 2.3346, 2.2981, 2.2628, 2.2287, 
2.1956, 2.1635, 2.1324, 2.1022, 2.0730, 2.0445, 2.0169, 1.9901, 
1.9640, 1.9386, 1.9139, 1.8899, 1.8665, 1.8437, 1.8214, 1.7998, 
1.7787, 1.7582, 1.7378, 1.7183, 1.6991, 1.6804, 1.6622, 1.6443, 
1.6269, 1.6098, 1.5931, 1.5768, 1.5608, 1.5452, 1.5299, 1.5149, 
1.5002, 1.4858, 1.4717, 1.4579, 1.4444, 1.4311, 1.4180, 1.4052, 
1.3925, 1.3800, 1.3672, 1.3470, 1.3480, 1.3352, 1.3237, 1.3125, 
1.3016, 1.2910, 1.2805, 1.2703, 1.2602, 1.2503, 1.2406, 1.2310, 
1.2217, 1.2124, 1.2033, 1.1944, 1.1856, 1.1770, 1.1685, 1.1601, 
1.1518, 1.1437, 1.1357, 1.1278, 1.1200, 1.1132, 1.1051, 1.0977, 
1.0904, 1.0832, 1.0761, 1.0691, 1.0623, 1.0555, 1.0488, 1.0422, 
1.0357, 1.0293, 1.0230, 1.0168, 1.0106, 1.0046, 0.9986, 0.9927, 
0.9868, 0.9811, 0.9754, 0.9697, 0.9641, 0.9583, 0.9508, 0.9505, 
0.9440, 0.9385, 0.9333, 0.9282, 0.9232, 0.9183, 0.9134, 0.9086, 
0.9038, 0.8992, 0.8945, 0.8900, 0.8855, 0.8810, 0.8766, 0.8723, 
0.8680, 0.8637, 0.8595, 0.8554, 0.8513, 0.8473, 0.8433, 0.8393, 
0.8378, 0.8315, 0.8277, 0.8239, 0.8202, 0.8165, 0.8128, 0.8092, 
0.8057, 0.8021, 0.7986, 0.7952, 0.7918, 0.7884, 0.7850, 0.7817, 
0.7785, 0.7752, 0.7720, 0.7689, 0.7657, 0.7627, 0.7596, 0.7566, 
0.7537, 0.7513, 0.7462, 0.7442, 0.7415, 0.7387, 0.7359, 0.7331, 
0.7304, 0.7276, 0.7249, 0.7222, 0.7196, 0.7169, 0.7143, 0.7117, 
0.7092, 0.7067, 0.7041, 0.7017, 0.6992, 0.6968, 0.6943, 0.6919, 
0.6895, 0.6872, 0.6848, 0.6815, 0.6807, 0.6783, 0.6760, 0.6737, 
0.6715, 0.6694, 0.6672, 0.6651, 0.6630, 0.6609, 0.6588, 0.6567, 
0.6547, 0.6527, 0.6507, 0.6487, 0.6467, 0.6448, 0.6429, 0.6410, 
0.6391, 0.6372, 0.6353, 0.6335, 0.6316, 0.6302, 0.6282, 0.6264, 
0.6246, 0.6229, 0.6211, 0.6194, 0.6177, 0.6161, 0.6144, 0.6127, 
0.6111, 0.6095, 0.6079, 0.6063, 0.6047, 0.6032, 0.6017, 0.6001, 
0.5986, 0.5972, 0.5957, 0.5943, 0.5930, 0.5919, 0.5927, 0.5861, 
0.5859, 0.5848, 0.5836, 0.5823, 0.5809, 0.5796, 0.5783, 0.5770, 
0.5757, 0.5744, 0.5731, 0.5718, 0.5705, 0.5693, 0.5680, 0.5668, 
0.5655, 0.5643, 0.5631, 0.5619, 0.5607, 0.5595, 0.5584, 0.5572, 
0.5577, 0.5549, 0.5538, 0.5527, 0.5515, 0.5504, 0.5494, 0.5483, 
0.5472, 0.5461, 0.5451, 0.5440, 0.5430, 0.5420, 0.5409, 0.5399, 
0.5389, 0.5379, 0.5369, 0.5359, 0.5350, 0.5340, 0.5330, 0.5320, 
0.5310, 0.5295, 0.5307, 0.5289, 0.5278, 0.5269, 0.5260, 0.5251, 
0.5242, 0.5233, 0.5225, 0.5216, 0.5208, 0.5199, 0.5191, 0.5183, 
0.5175, 0.5167, 0.5159, 0.5151, 0.5143, 0.5135, 0.5128, 0.5120, 
0.5112, 0.5104, 0.5096, 0.5079, 0.5087, 0.5078, 0.5070, 0.5063, 
0.5056, 0.5049, 0.5042, 0.5035, 0.5029, 0.5022, 0.5015, 0.5009, 
0.5003, 0.4996, 0.4990, 0.4984, 0.4977, 0.4971, 0.4965, 0.4959, 
0.4953, 0.4948, 0.4942, 0.4936, 0.4930, 0.4925, 0.4919, 0.4914, 
0.4908, 0.4903, 0.4898, 0.4892, 0.4887, 0.4882, 0.4877, 0.4872, 
0.4867, 0.4862, 0.4857, 0.4853, 0.4848, 0.4843, 0.4839, 0.4834, 
0.4830, 0.4826, 0.4822, 0.4818, 0.4815, 0.4813, 0.4826, 0.4782, 
0.4788, 0.4786, 0.4783, 0.4780, 0.4776, 0.4773, 0.4769, 0.4766, 
0.4762, 0.4758, 0.4755, 0.4751, 0.4748, 0.4745, 0.4741, 0.4738, 
0.4735, 0.4731, 0.4728, 0.4725, 0.4722, 0.4719, 0.4716, 0.4713, 
0.4712, 0.4707, 0.4704, 0.4702, 0.4699, 0.4696, 0.4694, 0.4691, 
0.4689, 0.4686, 0.4684, 0.4681, 0.4679, 0.4677, 0.4674, 0.4672, 
0.4670, 0.4668, 0.4666, 0.4664, 0.4661, 0.4659, 0.4657, 0.4655, 
0.4653, 0.4648, 0.4659, 0.4652, 0.4649, 0.4647, 0.4646, 0.4644, 
0.4642, 0.4641, 0.4640, 0.4638, 0.4637, 0.4636, 0.4634, 0.4633, 
0.4632, 0.4631, 0.4630, 0.4629, 0.4628, 0.4627, 0.4626, 0.4625, 
0.4624, 0.4623, 0.4621, 0.4602, 0.4628, 0.4625, 0.4624, 0.4623, 
0.4622, 0.4622, 0.4621, 0.4621, 0.4621, 0.4621, 0.4620, 0.4620, 
0.4620, 0.4620, 0.4621, 0.4621, 0.4621, 0.4621, 0.4621, 0.4622, 
0.4622, 0.4623, 0.4624, 0.4625, 0.4628, 0.4602, 0.4621, 0.4623, 
0.4624, 0.4625, 0.4626, 0.4627, 0.4628, 0.4629, 0.4630, 0.4631, 
0.4632, 0.4633, 0.4634, 0.4636, 0.4637, 0.4638, 0.4640, 0.4641, 
0.4642, 0.4644, 0.4646, 0.4647, 0.4649, 0.4652, 0.4659, 0.4647, 
0.4653, 0.4655, 0.4657, 0.4659, 0.4662, 0.4664, 0.4666, 0.4668, 
0.4670, 0.4672, 0.4674, 0.4677, 0.4679, 0.4681, 0.4684, 0.4686, 
0.4689, 0.4691, 0.4694, 0.4696, 0.4699, 0.4702, 0.4704, 0.4707, 
0.4712, 0.4713, 0.4716, 0.4719, 0.4722, 0.4725, 0.4728, 0.4731, 
0.4735, 0.4738, 0.4741, 0.4745, 0.4748, 0.4751, 0.4755, 0.4758, 
0.4762, 0.4766, 0.4769, 0.4773, 0.4776, 0.4780, 0.4783, 0.4786, 
0.4788, 0.4782, 0.4826, 0.4813, 0.4815, 0.4818, 0.4822, 0.4826, 
0.4830, 0.4834, 0.4839, 0.4843, 0.4848, 0.4853, 0.4857, 0.4862, 
0.4867, 0.4872, 0.4877, 0.4882, 0.4887, 0.4892, 0.4898, 0.4903, 
0.4908, 0.4914, 0.4919, 0.4925, 0.4930, 0.4936, 0.4942, 0.4948, 
0.4953, 0.4959, 0.4965, 0.4971, 0.4977, 0.4984, 0.4990, 0.4996, 
0.5003, 0.5009, 0.5015, 0.5022, 0.5029, 0.5035, 0.5042, 0.5049, 
0.5056, 0.5063, 0.5070, 0.5078, 0.5087, 0.5079, 0.5096, 0.5104, 
0.5112, 0.5120, 0.5128, 0.5135, 0.5143, 0.5151, 0.5159, 0.5167, 
0.5175, 0.5183, 0.5191, 0.5199, 0.5208, 0.5216, 0.5225, 0.5233, 
0.5242, 0.5251, 0.5260, 0.5269, 0.5278, 0.5289, 0.5307, 0.5295, 
0.5310, 0.5320, 0.5330, 0.5340, 0.5350, 0.5359, 0.5369, 0.5379, 
0.5389, 0.5399, 0.5409, 0.5420, 0.5430, 0.5440, 0.5451, 0.5461, 
0.5472, 0.5483, 0.5494, 0.5504, 0.5515, 0.5527, 0.5538, 0.5549, 
0.5577, 0.5572, 0.5584, 0.5595, 0.5607, 0.5619, 0.5631, 0.5643, 
0.5655, 0.5668, 0.5680, 0.5693, 0.5705, 0.5718, 0.5731, 0.5744, 
0.5757, 0.5770, 0.5783, 0.5796, 0.5809, 0.5823, 0.5836, 0.5848, 
0.5859, 0.5861, 0.5927, 0.5919, 0.5930, 0.5943, 0.5957, 0.5972, 
0.5986, 0.6001, 0.6017, 0.6032, 0.6047, 0.6063, 0.6079, 0.6095, 
0.6111, 0.6127, 0.6144, 0.6161, 0.6177, 0.6194, 0.6211, 0.6229, 
0.6246, 0.6264, 0.6282, 0.6302, 0.6316, 0.6335, 0.6353, 0.6372, 
0.6391, 0.6410, 0.6429, 0.6448, 0.6467, 0.6487, 0.6507, 0.6527, 
0.6547, 0.6567, 0.6588, 0.6609, 0.6630, 0.6651, 0.6672, 0.6694, 
0.6715, 0.6737, 0.6760, 0.6783, 0.6807, 0.6815, 0.6848, 0.6872, 
0.6895, 0.6919, 0.6943, 0.6967, 0.6992, 0.7017, 0.7041, 0.7066, 
0.7092, 0.7117, 0.7143, 0.7169, 0.7196, 0.7222, 0.7249, 0.7276, 
0.7304, 0.7331, 0.7359, 0.7387, 0.7415, 0.7442, 0.7462, 0.7513, 
0.7537, 0.7566, 0.7596, 0.7627, 0.7658, 0.7689, 0.7720, 0.7752, 
0.7785, 0.7817, 0.7850, 0.7884, 0.7918, 0.7952, 0.7986, 0.8021, 
0.8057, 0.8092, 0.8128, 0.8165, 0.8202, 0.8239, 0.8277, 0.8315, 
0.8378, 0.8393, 0.8433, 0.8473, 0.8513, 0.8554, 0.8595, 0.8637, 
0.8680, 0.8723, 0.8766, 0.8810, 0.8855, 0.8900, 0.8945, 0.8992, 
0.9038, 0.9086, 0.9134, 0.9183, 0.9232, 0.9282, 0.9333, 0.9385, 
0.9440, 0.9505, 0.9508, 0.9583, 0.9641, 0.9697, 0.9754, 0.9811, 
0.9868, 0.9927, 0.9986, 1.0046, 1.0106, 1.0168, 1.0230, 1.0293, 
1.0357, 1.0422, 1.0488, 1.0555, 1.0623, 1.0691, 1.0761, 1.0832, 
1.0904, 1.0977, 1.1051, 1.1132, 1.1200, 1.1278, 1.1357, 1.1437, 
1.1518, 1.1601, 1.1685, 1.1770, 1.1856, 1.1944, 1.2033, 1.2124, 
1.2217, 1.2310, 1.2406, 1.2503, 1.2602, 1.2703, 1.2805, 1.2910, 
1.3016, 1.3125, 1.3237, 1.3352, 1.3480, 1.3470, 1.3672, 1.3800, 
1.3925, 1.4052, 1.4180, 1.4311, 1.4444, 1.4579, 1.4717, 1.4858, 
1.5002, 1.5149, 1.5299, 1.5452, 1.5608, 1.5768, 1.5931, 1.6098, 
1.6269, 1.6443, 1.6622, 1.6804, 1.6991, 1.7183, 1.7378, 1.7582, 
1.7787, 1.7998, 1.8214, 1.8437, 1.8665, 1.8899, 1.9139, 1.9386, 
1.9640, 1.9901, 2.0169, 2.0445, 2.0730, 2.1022, 2.1324, 2.1635, 
2.1956, 2.2287, 2.2628, 2.2981, 2.3346, 2.3723, 2.4114, 2.4518, 
2.4949, 2.5372, 2.5823, 2.6291, 2.6777, 2.7283, 2.7810, 2.8359, 
2.8932, 2.9529, 3.0154, 3.0807, 3.1492, 3.2209, 3.2963, 3.3755, 
3.4589, 3.5469, 3.6398, 3.7381, 3.8423, 3.9529, 4.0707, 4.1963, 
4.3306, 4.4749, 4.6282, 4.7955, 4.9760, 5.1718, 5.3852, 5.6188, 
5.8755, 6.1594, 6.4749, 6.8281, 7.2262, 7.6787, 8.1980, 8.8004, 
9.5083, 10.3527, 11.3782, 12.6512, 14.2751, 16.4202, 19.3879, 23.7672, 
30.8839, 44.4752, 80.7389, 479.8513, 118.8994, 52.4252, 33.4815, 24.5580, 
19.4064, 16.0908, 13.8180, 12.2054, 11.0485, 10.2312, 9.6853, 9.3713, 
]

In [ ]:
freq = np.linspace(0,8000, 512)
hAx = plt.gca()
hAx.plot(freq, fft_mag[0:512])
hAx.set_xlim(0,400)
plt.grid(True)